# Занятие 8. Методы полей сил

In [ ]:
# 📥 Установка пакетов и библиотек
! pip install matplotlib
! pip install numpy
! pip install rdkit
! pip install parmed
! pip install scipy
! pip install seaborn

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

from rdkit import Chem
from rdkit.Chem import Draw, AllChem

import parmed as pmd

# show() открывает 3D-просмотр структур и траекторий GROMACS прямо под ячейкой
REPO = "/home/jupyter/project/computational-chemistry-course"
sys.path.append(os.path.join(REPO, "tools"))
from md_tools import show

In [ ]:
# 𓅨 Инициализация GROMACS и AmberTools
GMX = "/opt/gromacs/bin/gmx"
print("GROMACS найден:")
!{GMX} --version | head -1

AMBERHOME = "/opt/ambertools"                # AmberTools в образе курса: antechamber, parmchk2, tleap
os.environ["AMBERHOME"] = AMBERHOME
os.environ["PATH"] = os.environ["PATH"] + ":" + os.path.join(AMBERHOME, "bin")   # в конце PATH: python окружения AmberTools не перекрывает системный
!which antechamber

# 🗺️ Глава 1. Конвейер GROMACS и моделирование жидкости 

Расчёт в GROMACS проходит одну и ту же последовательность шагов независимо от моделируемой системы: координаты → топология → бокс → растворитель и ионы → минимизация энергии → уравновешивание NVT и NPT → продуктивный расчёт → анализ. 

Для каждого  этапа `grompp` объединяет координаты, топологию и параметры расчёта в двоичный файл `.tpr`, а `mdrun` выполняет расчёт по этому файлу.

| Файл | Содержимое |
|---|---|
| `.gro`, `.pdb` | координаты (и скорости в `.gro`), имена атомов и остатков, размер бокса |
| `.top`, `.itp` | топология: типы атомов, заряды, связи, углы, торсионы, состав системы |
| `.mdp` | параметры расчёта: интегратор, шаг, длительность, термостат, баростат |
| `.tpr` | всё вместе в двоичном виде, вход для `mdrun` |
| `.xtc`, `.trr` | траектория: сжатые координаты; координаты, скорости и силы с полной точностью |
| `.edr` | энергии, температура, давление, объём, плотность по времени |
| `.cpt` | контрольная точка: состояние системы, термостата и баростата для продолжения расчёта |
| `.log` | журнал `mdrun`: параметры, средние, производительность |
| `.xvg` | текстовый вывод программ анализа для построения графиков |


## 1.1 Подготовка 

In [ ]:
# Создаём папку и переходим в эту директорию
main_path = os.path.join(REPO, "Занятия", "Занятие_8")
work_path = os.path.join(main_path, "argon")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Бокс жидкого аргона: ребро из экспериментальной плотности, заполнение insert-molecules
N_AR  = 500             # число атомов в боксе
M_AR  = 39.948          # г/моль
RHO   = 1.407           # г/см³, жидкий аргон при 85 K
N_A   = 6.02214076e23
R_CUT = 1.0             # нм, радиус отсечки, который используем в .mdp

# Один атом в начале координат; бокс 1 нм здесь заглушка, размер задаст insert-molecules
ar_gro = """\
Argon
    1
    1AR      AR    1   0.000   0.000   0.000
   1.00000   1.00000   1.00000
"""
with open("ar.gro", "w") as f:
    f.write(ar_gro)

# Ребро кубического бокса из плотности и проверка против радиуса отсечки
V_nm3 = N_AR * M_AR / (RHO * N_A) * 1e21          # см³ → нм³
L = V_nm3 ** (1/3)
print(f"Объём бокса {V_nm3:.2f} нм³, ребро L = {L:.3f} нм, L/2 = {L/2:.3f} нм > r_cut = {R_CUT} нм: {L/2 > R_CUT}")

# Заполнение бокса случайными неперекрывающимися позициями
box = f"{L:.3f}"
!{GMX} insert-molecules -ci ar.gro -nmol {N_AR} -box {box} {box} {box} -try 50 -o box.gro > insert.log 2>&1

In [ ]:
show("box.gro", cmd="show spheres; set sphere_scale, 0.35; orient")

Аргон при 85К - одноатомная жидкость, поэтому в модели отсутствуют химические связи, валентные углы и частичные заряды. Межатомные взаимодействия описываются исключительно парным потенциалом Леннард–Джонса (LJ), который определяется двумя параметрами:

$$
U(r) = 4\varepsilon\left[
\left(\frac{\sigma}{r}\right)^{12}
-
\left(\frac{\sigma}{r}\right)^6
\right],
$$

где $r$ - расстояние между атомами, $\sigma$ и $\varepsilon$ - параметры LJ.

Для аргона используются следующие параметры ([Phys. Rev. 159, 98 (1967)](https://doi.org/10.1103/PhysRev.159.98)):

$$
\sigma = 3.405\ \text{Å}, \qquad
\frac{\varepsilon}{k_B} = 119.8\ \text{K},
$$

что соответствует $\varepsilon \approx 0.996\ \text{кДж/моль}$. 

In [ ]:
# Топология аргона
SIGMA_A = 3.405 
EPS_K   = 119.8            # K, ε/k_B

R_KJ     = 8.314462618e-3                  # кДж/(моль·K)
SIGMA_NM = SIGMA_A / 10                    # GROMACS хранит σ в нм
EPS_KJ   = EPS_K * R_KJ                    # и ε в кДж/моль: 119.8 K = 0.996 кДж/моль

# ParmEd хранит ЛД в единицах Amber: ε в ккал/моль и R_min/2 в Å, где R_min = 2^(1/6)·σ
eps_kcal    = EPS_KJ / 4.184
rmin_half_A = SIGMA_A * 2 ** (1 / 6) / 2

ar_type = pmd.AtomType("AR", 1, mass=M_AR, atomic_number=18)
ar_type.set_lj_params(eps_kcal, rmin_half_A)

atom = pmd.Atom(name="AR", type="AR", charge=0.0, mass=M_AR, atomic_number=18)
atom.atom_type = ar_type

one = pmd.Structure() # одна молекула = один атом в остатке AR
one.add_atom(atom, "AR", 1)
one.coordinates = [[0.0, 0.0, 0.0]]

top = pmd.gromacs.GromacsTopologyFile.from_structure(one * N_AR)   # 500 копий 
top.write("topol.top")

## 1.2 Минимизация энергии

In [ ]:
# Минимизация энергии: наискорейший спуск до максимальной силы < 100 кДж/моль/нм
em_mdp = """\
integrator    = steep        ; наискорейший спуск
emtol         = 100.0        ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep        = 0.01         ; начальный шаг, нм
nsteps        = 5000         ; предельное число шагов, если emtol не достигнут раньше

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = Cut-off      ; зарядов нет, параметр формальный
rcoulomb      = 1.0          ; нм
vdwtype       = Cut-off      ; LJ обрезается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм; должен быть меньше L/2 = 1.43 нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению за радиусом отсечки
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

!{GMX} grompp -f em.mdp -c box.gro -p topol.top -o em.tpr > grompp_em.out 2>&1
!{GMX} mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1

In [ ]:
# Потенциальная энергия по шагам минимизации
!printf "Potential\n\n" | {GMX} energy -f em.edr -o em_potential.xvg > energy_em.out 2>&1

In [ ]:
# Сходимость минимизации: E_pot по шагам и разность с конечной энергией в логарифмическом масштабе
data = np.loadtxt("em_potential.xvg", comments=("#", "@"))
step, epot = data.T

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12.5, 5))

ax1.plot(step, epot, "o", ms=3)
ax1.set_ylabel("E_pot, кДж/моль")
ax1.set_xlim(0, step[-1])

ax2.plot(step[:-1], epot[:-1] - epot.min(), "o", ms=3)
ax2.set_yscale("log")
ax2.set_ylabel("E_pot − E_min, кДж/моль")
ax2.set_xlabel("Шаг минимизации")
plt.show()

## 1.3 Уравновешивание в ансамбле NVT

После минимизации атомы находятся в локальном минимуме потенциальной энергии и неподвижны. Уравновешивание в ансамбле NVT переводит систему в равновесное состояние при заданной температуре до включения баростата. 

In [ ]:
# Уравновешивание NVT: 100 пс при 85 K, объём бокса фиксирован
nvt_mdp = """\
integrator    = md           ; Интегратор leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 50000        ; 50000 × 2 фс = 100 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = no           ; первый шаг динамики: стартовое состояние берётся из .gro
constraints   = none         ; без ограничений

cutoff-scheme = Verlet       ; список соседей Верле
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = Cut-off      ; электростатика с простой отсечкой (у аргона зарядов нет, параметр формальный)
rcoulomb      = 1.0          ; радиус отсечки электростатики, нм
vdwtype       = Cut-off      ; ЛД отсекается на rvdw без сглаживания
rvdw          = 1.0          ; радиус отсечки ЛД, нм (≈ 2.9 σ); должен быть меньше L/2 = 1.43 нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению за радиусом отсечки
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси: масштабирует скорости к целевой T + стохастическая поправка
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.5          ; время связи с термостатом, пс
ref-t         = 85           ; целевая температура, K
pcoupl        = no           ; объём фиксирован, ансамбль NVT

gen-vel       = yes          ; стартовые скорости берём из распределения Максвелла
gen-temp      = 85           ; стартовые скорости при температуре 85 K
gen-seed      = -1           ; seed для случайный чисел 
"""
with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp)

!{GMX} grompp -f nvt.mdp -c em.gro -p topol.top -o nvt.tpr > grompp_nvt.out 2>&1
!{GMX} mdrun -deffnm nvt -ntomp 4 > mdrun_nvt.out 2>&1

In [ ]:
# Энергии, температура и давление по времени NVT
!printf "Potential\nKinetic-En.\nTotal-Energy\nTemperature\nPressure\n\n" | {GMX} energy -f nvt.edr -o nvt_energy.xvg > energy_nvt.out 2>&1

In [ ]:
# NVT: энергия, температура и давление по времени
data = np.loadtxt("nvt_energy.xvg", comments=("#", "@"))
t, epot, ekin, etot, T, P = data.T                   # столбцы в порядке выбора термов

def hill(axd, y):
    pad = 3 * y.std() * len(y) ** -0.2
    grid = np.linspace(y.min() - pad, y.max() + pad, 300)
    kde = scipy.stats.gaussian_kde(y)(grid)
    axd.plot(kde / kde.max(), grid)
    axd.set_xlim(0, 1.05)
    axd.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    axd.spines[["top", "right"]].set_visible(True)

fig, axes = plt.subplots(3, 2, figsize=(12.5, 6.5), constrained_layout=False,
                         gridspec_kw={"width_ratios": [6, 1], "wspace": 0})
(ax1, d1), (ax2, d2), (ax3, d3) = axes
for ax in (ax2, ax3):
    ax.sharex(ax1)
for ax, d in axes:
    d.sharey(ax)

ax1.plot(t, etot)
ax1.set_ylabel("E, кДж/моль")
ax1.set_xlim(0, 100)
hill(d1, etot)

ax2.plot(t, T)
ax2.axhline(85, color="#D64933", lw=0.8, ls="--")   # целевая температура термостата
ax2.set_ylabel("T, K")
hill(d2, T)

ax3.plot(t, P)
ax3.axhline(1, color="#D64933", lw=0.8, ls="--")    # 1 бар: к нему приведёт баростат на шаге NPT
ax3.set_ylabel("P, бар")
ax3.set_xlabel("t, пс")
hill(d3, P)

for ax in (ax1, ax2):
    ax.tick_params(labelbottom=False)
plt.show()

## 1.4 Уравновешивание в NPT ансамбле 

Теперь уравновешивание с баростатом, изменяем размер бокса пока среднее давление не станет 1 бар. Берём C-rescale (стохастическое масштабирование ячейки): он быстро и без колебаний сводит давление к заданному и при этом даёт правильные флуктуации объёма.

In [ ]:
# Уравновешивание NPT: 100 пс при 85 K и 1 бар, продолжение NVT через .cpt
npt_mdp = """\
integrator    = md           ; leap-frog, как в NVT
dt            = 0.002        ; 2 фс
nsteps        = 50000        ; 50000 × 2 фс = 100 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии, T, P, объём и плотность в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = yes          ; продолжение NVT: скорости и состояние термостата из nvt.cpt
constraints   = none         ; без ограничений

cutoff-scheme = Verlet       ; список соседей Верле 
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = Cut-off      ; зарядов нет, параметр формальный
rcoulomb      = 1.0          ; радиус отсечки электростатики, нм
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм; бокс при сжатии останется больше 2·rvdw
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению (без неё баростат занизит плотность)
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы системы
tau-t         = 0.5          ; время связи с термостатом, пс
ref-t         = 85           ; целевая температура, K

pcoupl          = C-rescale  ; баростат, стохастическое масштабирование ячейки
pcoupltype      = isotropic  ; бокс сжимается и растягивается одинаково по трём осям: жидкость изотропна
tau-p           = 2.0        ; время связи с баростатом, пс (в 4 раза больше tau-t, чтобы объём не изменялся вместе с температурой)
ref-p           = 1.0        ; целевое давление, бар
compressibility = 2.0e-4     ; изотермическая сжимаемость жидкого Ar при 85 K, 1/бар (у воды 4.5e-5)

gen-vel       = no           ; скорости не генерируем, они уже в nvt.cpt

"""
with open("npt.mdp", "w") as f:
    f.write(npt_mdp)

!{GMX} grompp -f npt.mdp -c nvt.gro -t nvt.cpt -p topol.top -o npt.tpr > grompp_npt.out 2>&1
!{GMX} mdrun -deffnm npt -ntomp 4 > mdrun_npt.out 2>&1

In [ ]:
# Энергия, температура, давление, объём и плотность по времени NPT
!printf "Total-Energy\nTemperature\nPressure\nVolume\nDensity\n\n" | {GMX} energy -f npt.edr -o npt_energy.xvg > energy_npt.out 2>&1

In [ ]:
# NPT: энергия, температура, давление, объём и плотность по времени
data = np.loadtxt("npt_energy.xvg", comments=("#", "@"))
t, etot, T, P, V, rho = data.T                       # столбцы в порядке выбора термов

fig, axes = plt.subplots(5, 2, figsize=(12.5, 10), constrained_layout=False,
                         gridspec_kw={"width_ratios": [6, 1], "wspace": 0})
(ax1, d1), (ax2, d2), (ax3, d3), (ax4, d4), (ax5, d5) = axes
for ax in (ax2, ax3, ax4, ax5):
    ax.sharex(ax1)
for ax, d in axes:
    d.sharey(ax)

ax1.plot(t, etot)
ax1.set_ylabel("E, кДж/моль")
ax1.set_xlim(0, 100)
hill(d1, etot)

ax2.plot(t, T)
ax2.axhline(85, color="#D64933", lw=0.8, ls="--")   # целевая температура термостата
ax2.set_ylabel("T, K")
hill(d2, T)

ax3.plot(t, P)
ax3.axhline(1, color="#D64933", lw=0.8, ls="--")    # целевое давление баростата
ax3.set_ylabel("P, бар")
hill(d3, P)

ax4.plot(t, V)
ax4.set_ylabel("V, нм³")
hill(d4, V)

ax5.plot(t, rho)
ax5.axhline(1407, color="#D64933", lw=0.8, ls="--") # эксперимент: 1.407 г/см³ при 85 K, 1 бар
ax5.set_ylabel("ρ, кг/м³")
ax5.set_xlabel("t, пс")
hill(d5, rho)

for ax in (ax1, ax2, ax3, ax4):
    ax.tick_params(labelbottom=False)
plt.show()

## 1.5 Моделирование в NPT ансамбле 

Длинное моделирование идёт с теми же параметрами, что и NPT-уравновешивание, но траектория стартует из уравновешенного состояния и может использоваться для анализа.

In [ ]:
# Продуктивный расчёт NPT: 1 нс при 85 K и 1 бар
md_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 500000       ; 500000 × 2 фс = 1 нс

nstxout-compressed = 5000    ; координаты в md.xtc каждые 10 пс: 100 кадров для просмотра
nstenergy     = 500          ; термы в md.edr каждую 1 пс
nstlog        = 5000         ; строка в md.log каждые 10 пс

continuation  = yes          ; продолжение NPT: скорости и состояние термостата и баростата из npt.cpt
constraints   = none         ; связей нет, фиксировать нечего

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = Cut-off      ; зарядов нет, параметр формальный
rcoulomb      = 1.0          ; радиус отсечки электростатики, нм
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению за радиусом отсечки
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что при уравновешивании
tc-grps       = System       ; одна термостатируемая группа: все атомы
tau-t         = 0.5          ; время связи с термостатом, пс
ref-t         = 85           ; целевая температура, K

pcoupl          = C-rescale  ; баростат тот же, что при уравновешивании: правильные флуктуации объёма
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 2.0e-4     ; изотермическая сжимаемость жидкого Ar при 85 K, 1/бар

gen-vel       = no           ; скорости не генерируем, они в npt.cpt
"""
with open("md.mdp", "w") as f:
    f.write(md_mdp)

!{GMX} grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr > grompp_md.out 2>&1
!{GMX} mdrun -deffnm md -ntomp 4 > mdrun_md.out 2>&1

In [ ]:
# Энергия, температура, давление, объём и плотность
!printf "Total-Energy\nTemperature\nPressure\nVolume\nDensity\n\n" | {GMX} energy -f md.edr -o md_energy.xvg > energy_md.out 2>&1

In [ ]:
# Продуктивный расчёт: энергия, температура, давление, объём и плотность по времени; справа распределения
data = np.loadtxt("md_energy.xvg", comments=("#", "@"))
t, etot, T, P, V, rho = data.T                       # столбцы в порядке выбора термов

fig, axes = plt.subplots(5, 2, figsize=(12.5, 10), constrained_layout=False,
                         gridspec_kw={"width_ratios": [6, 1], "wspace": 0})
(ax1, d1), (ax2, d2), (ax3, d3), (ax4, d4), (ax5, d5) = axes
for ax in (ax2, ax3, ax4, ax5):
    ax.sharex(ax1)
for ax, d in axes:
    d.sharey(ax)

ax1.plot(t, etot)
ax1.set_ylabel("E, кДж/моль")
ax1.set_xlim(0, 1000)
hill(d1, etot)

ax2.plot(t, T)
ax2.axhline(85, color="#D64933", lw=0.8, ls="--")   # целевая температура термостата
ax2.set_ylabel("T, K")
hill(d2, T)

ax3.plot(t, P)
ax3.axhline(1, color="#D64933", lw=0.8, ls="--")    # целевое давление баростата
ax3.set_ylabel("P, бар")
hill(d3, P)

ax4.plot(t, V)
ax4.set_ylabel("V, нм³")
hill(d4, V)

ax5.plot(t, rho)
ax5.axhline(1407, color="#D64933", lw=0.8, ls="--") # эксперимент: 1.407 г/см³ при 85 K
ax5.set_ylabel("ρ, кг/м³")
ax5.set_xlabel("t, пс")
hill(d5, rho)

for ax in (ax1, ax2, ax3, ax4):
    ax.tick_params(labelbottom=False)
plt.show()

In [ ]:
show("md.gro", "md.xtc", cmd="show spheres; set sphere_scale, 0.35; orient")

## 1.6 Расчёт энтальпии испарения

Энтальпия испарения это разность энтальпий газа и жидкости, $H = \langle E_\mathrm{кин}\rangle + \langle U\rangle + pV$. При одной температуре кинетическая энергия в обеих фазах одинакова и сокращается. Газообразный аргон считаем идеальным газом: $\langle U\rangle_\mathrm{газ} = 0$, $pV_\mathrm{газ} = RT$. Остаётся
$$\Delta H_\mathrm{исп} \approx -\frac{\langle U\rangle}{N} + RT,$$

где $\langle U\rangle$ это среднее значений Potential.

In [ ]:
# Потенциальная энергия
!printf "Potential\n\n" | {GMX} energy -f md.edr -o md_potential.xvg > energy_pot.out 2>&1

In [ ]:
# Энтальпия испарения из потенциальной энергии: ΔH_исп = -⟨U⟩/N + RT
N_AR = 500                  # число атомов в боксе
R    = 8.314462618e-3       # кДж/(моль·K)
T    = 85.0                 # K, ref-t термостата

t, U = np.loadtxt("md_potential.xvg", comments=("#", "@")).T
dH = -U / N_AR + R * T                                               # мгновенная ΔH_исп в каждом кадре
blocks = dH[: len(dH) // 10 * 10].reshape(10, -1).mean(axis=1)       # 10 блоков по 100 пс
err = blocks.std(ddof=1) / np.sqrt(10)

print(f"⟨U⟩/N              = {U.mean() / N_AR:.3f} кДж/моль")
print(f"RT                 = {R * T:.3f} кДж/моль")
print(f"ΔH_исп             = {dH.mean():.3f} ± {err:.3f} кДж/моль")

# 💧 Глава 2. Моделирование чистых жидкостей

Параметры для стандартных типов воды SPC, SPC/E, TIP3P, TIP4P, TIP4P-Ew, TIP5P уже лежат в каталоге большинства силовых полей GROMACS отдельными `.itp`. 

## 2.1 Подготовка

In [ ]:
# Модели воды в силовом поле OPLS-AA
GMXTOP = os.path.join(os.path.dirname(os.path.dirname(GMX)), "share", "gromacs", "top")
FF = "oplsaa.ff"

models = sorted(f[:-4] for f in os.listdir(os.path.join(GMXTOP, FF)) if f.endswith(".itp") and f.startswith(("spc", "tip")))
print("Модели воды:", *models)
print("Равновесные боксы:", *sorted(f for f in os.listdir(GMXTOP) if f.endswith(".gro")))

In [ ]:
# Создаём папку и переходим в эту директорию
main_path = os.path.join(REPO, "Занятия", "Занятие_8")
work_path = os.path.join(main_path, "water")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Бокс воды SPC/E и топология
BOX = "spc216"
MODEL = "spce"
L = 2.5                                              # нм; ≈ 500 молекул, отсечка 1.0 нм < L/2
!{GMX} solvate -cs {BOX} -box {L} {L} {L} -o water.gro > solvate.log 2>&1
!grep -E "Number of solvent molecules|Output configuration" solvate.log

for line in open("solvate.log"):                     # число молекул берём из лога, а не пишем руками
    if "Number of solvent molecules" in line:
        n_sol = int(line.split()[-1])

topol_top = f"""\
#include "oplsaa.ff/forcefield.itp"                 ; параметры ЛД и правила комбинирования
#include "oplsaa.ff/{MODEL}.itp"                    ; модель воды: заряды, геометрия, SETTLE

[ system ]
SPC/E water

[ molecules ]
SOL   {n_sol}
"""
with open("topol.top", "w") as f:
    f.write(topol_top)

In [ ]:
show("water.gro")

## 2.2 Протокол моделирования 

Симуляция бокса с водой отличается от аргона:

* **Электростатика.** На атомах воды стоят заряды, кулоновское взаимодействие убывает медленно, как 1/r, и простая отсечка на 1 нм искажает и структуру, и давление. Поэтому электростатику считаем методом PME.

* **Силовое поле.** Файл `forcefield.itp` из OPLS-AA подключён только ради `[ defaults ]` и типов `opls_116`/`opls_117`: из них берутся σ и ε кислорода, у водородов LJ нет, а заряды задаёт `spce.itp`.

In [ ]:
# Минимизация энергии воды: наискорейший спуск, электростатика PME
em_mdp = """\
integrator    = steep        ; наискорейший спуск, как в главе 1
emtol         = 100.0        ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep        = 0.01         ; начальный шаг, нм
nsteps        = 5000         ; предельное число шагов, если emtol не достигнут раньше

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду: прямая часть до rcoulomb, остальное в обратном пространстве
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм; меньше L/2 = 1.25 нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм; в схеме Verlet обычно равен rcoulomb
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению за радиусом отсечки
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

!{GMX} grompp -f em.mdp -c water.gro -p topol.top -o em.tpr > grompp_em.out 2>&1
!{GMX} mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1

In [ ]:
# Уравновешивание NVT: 100 пс при 298.15 K, объём бокса фиксирован
nvt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: молекула жёсткая (SETTLE), быстрых колебаний O-H в динамике нет
nsteps        = 50000        ; 50000 × 2 фс = 100 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = no           ; первый шаг динамики: стартовое состояние из em.gro
constraints   = none         ; связей в топологии нет, геометрию воды держит [ settles ] из spce.itp

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению за радиусом отсечки
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, как в главе 1
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс; стандарт для воды
ref-t         = 298.15       ; целевая температура, K
pcoupl        = no           ; объём фиксирован, ансамбль NVT

gen-vel       = yes          ; стартовые скорости из распределения Максвелла
gen-temp      = 298.15       ; при температуре 298.15 K
gen-seed      = -1           ; seed для случайных чисел
"""
with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp)

!{GMX} grompp -f nvt.mdp -c em.gro -p topol.top -o nvt.tpr > grompp_nvt.out 2>&1
!{GMX} mdrun -deffnm nvt -ntomp 4 > mdrun_nvt.out 2>&1

In [ ]:
# NPT одним запуском, 200 пс при 298.15 K и 1 бар, три варианта: жёсткая 2 фс, гибкая 2 фс, гибкая 0.5 фс
runs = [
    # (tag,        шаг, пс, define)
    ("md",        0.002,  ""),
    ("md_flex2",  0.002,  "-DFLEXIBLE"),
    ("md_flex05", 0.0005, "-DFLEXIBLE"),
]

for tag, dt, define in runs:
    nsteps = int(round(200 / dt))           # 200 пс: первые 100 пс - уравновешивание, в анализе отбрасываем
    nstxtc = int(round(1 / dt))             # кадр в .xtc каждую 1 пс
    nstenergy = int(round(0.2 / dt))        # .edr каждые 0.2 пс
    nstlog = int(round(2 / dt))             # .log каждые 2 пс
    geom = "ограничений нет: связи O-H и угол H-O-H колеблются" if define else "геометрию воды держит SETTLE из spce.itp"
    mdp = f"""\
define        = {define:<12} ; -DFLEXIBLE: в spce.itp вместо [ settles ] гармонические связи O-H и угол H-O-H; пусто - жёсткая вода
integrator    = md           ; leap-frog
dt            = {dt:<12} ; шаг интегрирования, пс
nsteps        = {nsteps:<12} ; {nsteps} × {dt * 1000:g} фс = 200 пс

nstxout-compressed = {nstxtc:<7} ; кадр в {tag}.xtc каждую 1 пс, траектория для анализа
nstenergy     = {nstenergy:<12} ; энергии, T, P, объём и плотность в .edr каждые 0.2 пс
nstlog        = {nstlog:<12} ; строка в .log каждые 2 пс

continuation  = yes          ; продолжение NVT: скорости и состояние термостата из nvt.cpt
constraints   = none         ; {geom}

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению за радиусом отсечки
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, стохастическое масштабирование ячейки, как в главе 1
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости не генерируем, они уже в nvt.cpt
"""
    with open(f"{tag}.mdp", "w") as f:
        f.write(mdp)

    print(f"=== {tag}: шаг {dt * 1000:g} фс, {nsteps} шагов ===")
    !{GMX} grompp -f {tag}.mdp -c nvt.gro -t nvt.cpt -p topol.top -o {tag}.tpr > grompp_{tag}.out 2>&1
    !{GMX} mdrun -deffnm {tag} -ntomp 4 > mdrun_{tag}.out 2>&1

In [ ]:
# Три расчёта воды SPC: дрейф сохраняемой энергии, температура и плотность
tags = ["md", "md_flex2", "md_flex05"]
labels = ["жёсткая, 2 фс", "гибкая, 2 фс", "гибкая, 0.5 фс"]
N = 510                                              # число молекул воды в ячейке

for tag in tags:
    !printf "Conserved-En.\nTemperature\nDensity\n\n" | {GMX} energy -f {tag}.edr -o {tag}_cmp.xvg > energy_{tag}_cmp.out 2>&1

data = {tag: np.loadtxt(f"{tag}_cmp.xvg", comments=("#", "@")) for tag in tags}

fig, axes = plt.subplots(3, 2, figsize=(12.5, 7.5), constrained_layout=False,
                         gridspec_kw={"width_ratios": [6, 1], "wspace": 0})
(ax1, d1), (ax2, d2), (ax3, d3) = axes
for ax in (ax2, ax3):
    ax.sharex(ax1)
for ax, d in axes:
    d.sharey(ax)

for tag, label in zip(tags, labels):
    t, econs, T, rho = data[tag].T                   # столбцы в порядке термов в .edr
    ax1.plot(t, (econs - econs[0]) / N, label=label)
    ax2.plot(t, T, lw=0.8)
    hill(d2, T[t >= 100])                            # распределения - по продуктивному участку 100-200 пс
    ax3.plot(t, rho, lw=0.8)
    hill(d3, rho[t >= 100])

ax1.set_ylabel("ΔE, кДж/моль\nна молекулу")
ax1.set_xlim(0, 200)
ax1.legend(frameon=False)
d1.axis("off")                                       # дрейф - не флуктуация: распределение здесь ничего не говорит

ax2.axhline(298.15, color="#D64933", lw=0.8, ls="--", zorder=3)  # целевая температура термостата
ax2.set_ylabel("T, K")
ax2.set_ylim(265, 335)                               # стартовые выбросы гибких моделей (190-350 K) уходят за край

ax3.axhline(997, color="#D64933", lw=0.8, ls="--", zorder=3)     # эксперимент: 997 кг/м³ при 298 K, 1 бар
ax3.set_ylabel("ρ, кг/м³")
ax3.set_ylim(940, 1120)                              # стартовый провал до 860 кг/м³ (гибкая, 2 фс) уходит за край
ax3.set_xlabel("t, пс")

for ax in (ax1, ax2, ax3):
    ax.axvline(100, color="0.6", lw=0.8)             # граница уравновешивания
for ax in (ax1, ax2):
    ax.tick_params(labelbottom=False)
fig.align_ylabels(axes[:, 0])
plt.show()

# Продуктивный участок 100-200 пс: дрейф сохраняемой энергии, средние T и ρ
print(f"{'расчёт':<16}{'дрейф, кДж/моль/нс':>20}{'<T>, K':>9}{'<ρ>, кг/м³':>14}")
for tag, label in zip(tags, labels):
    t, econs, T, rho = data[tag].T
    m = t >= 100
    drift = np.polyfit(t[m], econs[m], 1)[0] * 1000 / N   # наклон, кДж/моль/пс → на молекулу за нс
    print(f"{label:<16}{drift:>20.1f}{T[m].mean():>9.1f}{rho[m].mean():>9.0f} ± {rho[m].std():.0f}")

## 2.3 Структура раствора 

Радиальная функция распределения (RDF) показывает, во сколько раз плотность частиц типа B на расстоянии r от частицы типа A больше средней:

$$g_{AB}(r) = \frac{\langle n_B(r, r+\Delta r)\rangle}{4\pi r^2 \Delta r \, \rho_B},$$

где $n_B$ число частиц B в сферическом слое, $\rho_B$ их средняя плотность в боксе. 

Интеграл RDF даёт координационное число $n(r) = 4\pi\rho_B\int_0^r g(r')\,r'^2\,dr'$, среднее число соседей внутри сферы радиуса r. Координационное число рассчитанное в первом минимуме g(r) принимают за число молекул в первой гидратной оболочке. Флаг `-excl` убирает пары атомов одной молекулы (иначе в g_OH и g_HH появятся пики на 0.100 и 0.163 нм), `-rmax` ограничиваем 1.0 нм, меньше половины ребра бокса.

In [ ]:
# Радиальные функции распределения O-O, O-H, H-H
!{GMX} rdf -f md.xtc -s md.tpr -ref "name OW" -sel "name OW" "name HW1 HW2" -excl -bin 0.002 -rmax 1.0 -o rdf_O.xvg -cn rdf_O_cn.xvg > rdf_O.out 2>&1
!{GMX} rdf -f md.xtc -s md.tpr -ref "name HW1 HW2" -sel "name HW1 HW2" -excl -bin 0.002 -rmax 1.0 -o rdf_H.xvg > rdf_H.out 2>&1

In [ ]:
# g(r) для пар O-O, O-H, H-H
r, g_oo, g_oh = np.loadtxt("rdf_O.xvg", comments=("#", "@")).T
r_h, g_hh = np.loadtxt("rdf_H.xvg", comments=("#", "@")).T
r_cn, n_oo, n_oh = np.loadtxt("rdf_O_cn.xvg", comments=("#", "@")).T

fig, ax = plt.subplots(figsize=(12.5, 5))
ax.plot(r, g_oo, color="black", label="O-O")
ax.plot(r, g_oh, color="red", label="O-H")
ax.plot(r_h, g_hh, color="blue", label="H-H")
ax.set_xticks(np.arange(0.1, 1.01, 0.05))
ax.set_xlim(r[np.argmax(g_oh > 0)], 1.0)
ax.set_ylim(0, None)
ax.set_xlabel("r, нм")
ax.set_ylabel("g(r)")
ax.set_title("Радиальные функции распределения воды SPC/E, 298.15 K, 1 бар", fontsize=12)
ax.legend(frameon=False)
plt.show()

# 🧪 Глава 3. Моделирование многокомпонентных систем

## 3.1 Хлорид натрия в воде

В отличие от раствора чистой воды в топологию добавляется ``#include "oplsaa.ff/ions.itp"``: там описаны параметры для одноатомных молекул NA и CL с зарядами ±1 и параметрами LJ, а смешанные пары Леннарда-Джонса Na–O, Cl–O и Na–Cl получаются по правилу комбинирования силового поля (§ 1.5.2, Занятие 7). 

Ионы можно добавить в систему используя ``gmx genion``, который заменяет случайно выбранные молекулы воды на ионы и сам исправляет блок ``[ molecules ]`` в ``topol.top``. Число катионов и анионов берём одинаковым чтобы получить нулевой заряд системы. 9 пар ионов и 492 молекулы воды дают молярную концентрацию ~ 1 моль/л.

### 3.1.1 Подготовка

In [ ]:
# Создаём папку и переходим в эту директорию
main_path = os.path.join(REPO, "Занятия", "Занятие_8")
work_path = os.path.join(main_path, "NaCl")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Бокс воды SPC/E и топология с ионами: 9 пар Na⁺/Cl⁻ на ≈ 500 молекул воды дают раствор ≈ 1 моль/кг
L = 2.5                                              # ребро бокса, нм; отсечка 1.0 нм < L/2
N_ION = 9                                            # число пар Na⁺/Cl⁻
!{GMX} solvate -cs spc216 -box {L} {L} {L} -o water.gro > solvate.log 2>&1
!grep -E "Number of solvent molecules|Output configuration" solvate.log

for line in open("solvate.log"):                     # число молекул берём из лога, а не пишем руками
    if "Number of solvent molecules" in line:
        n_sol = int(line.split()[-1])

# ions.itp
topol_top = f"""\
#include "oplsaa.ff/forcefield.itp"                 ; параметры ЛД и правила комбинирования
#include "oplsaa.ff/spce.itp"                      ; модель воды SPC/E: заряды, геометрия, SETTLE
#include "oplsaa.ff/ions.itp"                      ; одноатомные ионы, среди них NA и CL

[ system ]
NaCl in SPC/E water

[ molecules ]
SOL   {n_sol}
"""
with open("topol.top", "w") as f:
    f.write(topol_top)

# Концентрация: genion заменит 2·N_ION молекул воды на ионы
M_W = 0.018015                                       # молярная масса воды, кг/моль
N_A = 6.02214076e23                                  # число Авогадро, 1/моль
n_w = n_sol - 2 * N_ION                              # молекул воды после genion
molality = N_ION / (n_w * M_W)                       # моль NaCl на кг воды
molarity = N_ION / N_A / (L**3 * 1e-24)              # моль на литр; 1 нм³ = 1e-24 л
print(f"Молекул воды: {n_sol} → {n_w} после замены на {2 * N_ION} ионов")
print(f"Молярная концентрация: {molarity:.3f} моль/л (объём стартового бокса {L**3:.3f} нм³)")

In [ ]:
# Добавление ионов (параметры из ions.mdp почти формальные)
ions_mdp = """\
integrator    = steep        ; наискорейший спуск, как в минимизации
nsteps        = 5000         ; ions.tpr нужен только genion, mdrun по нему не запускаем
cutoff-scheme = Verlet       ; список соседей Верле с буфером
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("ions.mdp", "w") as f:
    f.write(ions_mdp)

!{GMX} grompp -f ions.mdp -c water.gro -p topol.top -o ions.tpr > grompp_ions.out 2>&1
!echo SOL | {GMX} genion -s ions.tpr -o nacl.gro -p topol.top -pname NA -nname CL -np {N_ION} -nn {N_ION} > genion.log 2>&1

### 3.1.2 Моделирование 

In [ ]:
# Минимизация энергии раствора
em_mdp = """\
define        = -DFLEXIBLE   ; гибкая вода только на минимизации: при жёсткой SETTLE не справляется с большими шагами steep и сыплет step*.pdb
integrator    = steep        ; наискорейший спуск, как в главах 1-2
emtol         = 100.0        ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep        = 0.01         ; начальный шаг, нм
nsteps        = 5000         ; предельное число шагов

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду, как в главе 2
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм; меньше L/2 = 1.25 нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

!{GMX} grompp -f em.mdp -c nacl.gro -p topol.top -o em.tpr > grompp_em.out 2>&1
!{GMX} mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1

In [ ]:
# Уравновешивание NVT: 20 пс при 298.15 K, объём бокса фиксирован
nvt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: вода жёсткая (SETTLE), ионы одноатомные
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = no           ; первый шаг динамики: стартовое состояние из em.gro
constraints   = none         ; геометрию воды держит [ settles ] из spce.itp

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, как в главе 2
tc-grps       = System       ; одна группа: вода и ионы вместе
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K
pcoupl        = no           ; объём фиксирован, ансамбль NVT

gen-vel       = yes          ; стартовые скорости из распределения Максвелла
gen-temp      = 298.15       ; при температуре 298.15 K
gen-seed      = -1           ; seed для случайных чисел
"""
with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp)

!{GMX} grompp -f nvt.mdp -c em.gro -p topol.top -o nvt.tpr > grompp_nvt.out 2>&1
!{GMX} mdrun -deffnm nvt -ntomp 4 > mdrun_nvt.out 2>&1

In [ ]:
# Уравновешивание NPT: 20 пс при 298.15 K и 1 бар
npt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии, объём и плотность в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = yes          ; продолжение NVT: скорости и состояние термостата из nvt.cpt
constraints   = none         ; геометрию воды держит [ settles ] из spce.itp

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, стохастическое масштабирование ячейки; годится и для уравновешивания
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости не генерируем, они уже в nvt.cpt
"""
with open("npt.mdp", "w") as f:
    f.write(npt_mdp)

!{GMX} grompp -f npt.mdp -c nvt.gro -t nvt.cpt -p topol.top -o npt.tpr > grompp_npt.out 2>&1
!{GMX} mdrun -deffnm npt -ntomp 4 > mdrun_npt.out 2>&1

In [ ]:
# Моделирование NPT: 1 нс при 298.15 K и 1 бар, весь расчёт идёт в анализ
md_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 500000       ; 500000 × 2 фс = 1 нс

nstxout-compressed = 500     ; кадр в md.xtc каждую 1 пс: 1000 кадров для RDF
nstenergy     = 500          ; энергии, T, P, объём и плотность в .edr каждую 1 пс: 1000 точек для средних
nstlog        = 5000         ; строка в .log каждые 10 пс

continuation  = yes          ; продолжение NPT: скорости, термостат и баростат из npt.cpt
constraints   = none         ; геометрию воды держит [ settles ] из spce.itp

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению за радиусом отсечки
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, тот же, что при уравновешивании
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар; у 1 М раствора чуть меньше

gen-vel       = no           ; скорости не генерируем, они уже в npt.cpt
"""
with open("md.mdp", "w") as f:
    f.write(md_mdp)

!{GMX} grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr > grompp_md.out 2>&1
!{GMX} mdrun -deffnm md -ntomp 4 > mdrun_md.out 2>&1

In [ ]:
show("md.gro", "md.xtc", stride=10, cmd="show spheres; set sphere_scale, 0.35; orient")

In [ ]:
# Температура, давление и плотность раствора по времени; справа распределения по всему продуктивному расчёту 0-200 пс
!printf "Temperature\nPressure\nDensity\n\n" | {GMX} energy -f md.edr -o md_tpd.xvg > energy_md.out 2>&1

t, T, P, rho = np.loadtxt("md_tpd.xvg", comments=("#", "@")).T   # столбцы в порядке термов в .edr

fig, axes = plt.subplots(3, 2, figsize=(12.5, 6.5), constrained_layout=False,
                         gridspec_kw={"width_ratios": [6, 1], "wspace": 0})
(ax1, d1), (ax2, d2), (ax3, d3) = axes
for ax in (ax2, ax3):
    ax.sharex(ax1)
for ax, d in axes:
    d.sharey(ax)

ax1.plot(t, T)
ax1.axhline(298.15, color="#D64933", lw=0.8, ls="--")   # целевая температура термостата
ax1.set_ylabel("T, K")
ax1.set_xlim(0, t[-1])
hill(d1, T)

ax2.plot(t, P)
ax2.axhline(1, color="#D64933", lw=0.8, ls="--")        # целевое давление баростата
ax2.set_ylabel("P, бар")
hill(d2, P)

ax3.plot(t, rho)
ax3.axhline(1036, color="#D64933", lw=0.8, ls="--")  # эксперимент: 1036 кг/м³
ax3.set_ylabel("ρ, кг/м³")
ax3.set_xlabel("t, пс")
hill(d3, rho)

for ax in (ax1, ax2):
    ax.tick_params(labelbottom=False)
plt.show()

### 3.1.3 Гидратная оболочка и гидратные числа

Ион упорядочивает окружающие молекулы воды в гидратные оболочки, которые проявляются максимумами функции радиального распределения $g(r)$. Число молекул в первой оболочке (гидратное число) равно текущему координационному числу в первом минимуме $g(r)$:

$$n(r_{min}) = 4\pi\rho_O \int_0^{r_{min}} g(r)\,r^2\,dr,$$

где $\rho_O$ — числовая плотность атомов кислорода воды.

Ориентация молекул воды в первой оболочке определяется знаком заряда иона. К катиону Na⁺ молекула обращена атомом кислорода, поэтому пик Na-O узкий и расположен на малом расстоянии. Анион Cl⁻ является акцептором водородных связей: молекула воды обращена к нему атомом водорода, и пик Cl-H расположен примерно на 0.1 нм ближе пика Cl-O.

In [ ]:
# Радиальные функции распределения
!{GMX} rdf -f md.xtc -s md.tpr -ref "name NA" -sel "name OW" "name CL" -bin 0.002 -rmax 1.0 -o rdf_Na.xvg > rdf_Na.out 2>&1
!{GMX} rdf -f md.xtc -s md.tpr -ref "name CL" -sel "name OW" "name HW1 HW2" -bin 0.002 -rmax 1.0 -o rdf_Cl.xvg > rdf_Cl.out 2>&1

In [ ]:
# g(r) вокруг Na⁺ и Cl⁻; первый максимум, первый минимум и гидратное число n(r_min)
r, g_na_o, g_na_cl = np.loadtxt("rdf_Na.xvg", comments=("#", "@")).T     # столбцы в порядке -sel: OW, CL
_, g_cl_o, g_cl_h = np.loadtxt("rdf_Cl.xvg", comments=("#", "@")).T      # столбцы в порядке -sel: OW, HW1 HW2

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(12.5, 7), gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(r, g_na_o, color="black", label="Na-O")
ax1.plot(r, g_cl_o, color="blue", label="Cl-O")
ax1.plot(r, g_cl_h, color="red", label="Cl-H")
ax1.axhline(1, color="0.6", lw=0.5)
ax1.set_ylabel("g(r)")
ax1.legend(frameon=False)


ax1.set_xticks(np.arange(0, 0.81, 0.05), minor=True)
ax1.set_xlim(r[np.argmax(g_cl_h > 0)], 0.8)
ax1.set_ylim(0, 8)
ax1.set_xlabel("r, нм")
ax1.set_ylabel("g(r)")

ax2.plot(r, g_na_cl, color="#FF674D", label="Na-Cl")
ax2.axhline(1, color="0.6", lw=0.5)
ax2.set_ylabel("g(r)")
ax2.set_xlabel("r, нм")
ax2.legend(frameon=False)

ax2.set_xticks(np.arange(0, 0.81, 0.05), minor=True)
ax2.set_xlim(r[np.argmax(g_cl_h > 0)], 0.8)
ax2.set_ylim(0, 1.05 * g_na_cl.max())
ax2.set_xlabel("r, нм")
ax2.set_ylabel("g(r)")

plt.show()

## 3.2 Раствор этанола и воды

Этанол смешивается с водой в любых пропорциях, но объём смеси меньше суммы объёмов компонентов, а при смешении выделяется тепло. Обе величины, избыточный объём и энтальпию смешения, можно получить из средних по NPT-расчётам нескольких составов. 

Для молекул этанола нужна топология и параметры, а их нет в базе остатков GROMACS, поэтому строим сами из силового поля для малых молекул GAFF2.

### 3.2.1 Подготовка системы

In [ ]:
# Создаём папку и переходим в эту директорию
main_path = os.path.join(REPO, "Занятия", "Занятие_8")
work_path = os.path.join(main_path, "ethanol")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Молекула этанола из SMILES: 3D-геометрия RDKit, остаток ETH, имена атомов для топологии
mol = Chem.AddHs(Chem.MolFromSmiles("CCO"))
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

# RDKit по умолчанию пишет остаток UNL и водород гидроксила как H6; задаём остаток ETH и имя HO
counts = {}
for atom in mol.GetAtoms():
    el = atom.GetSymbol()
    counts[el] = counts.get(el, 0) + 1
    name = f"{el}{counts[el]}"
    if el == "H" and atom.GetNeighbors()[0].GetSymbol() == "O":
        name = "HO"                                  # водород гидроксила: донор водородной связи в 3.2.2
    info = Chem.AtomPDBResidueInfo()
    info.SetName(f" {name:<3s}")                     # PDB: имя атома в колонках 13-16, элемент с 14-й
    info.SetResidueName("ETH")
    info.SetResidueNumber(1)
    atom.SetMonomerInfo(info)

Chem.MolToPDBFile(mol, "ethanol.pdb")
!grep ATOM ethanol.pdb

In [ ]:
# Этанол в GAFF2 с зарядами AM1-BCC: antechamber → parmchk2 → tleap → ParmEd → ethanol.itp и atomtypes_eth.itp
!antechamber -i ethanol.pdb -fi pdb -o ethanol.mol2 -fo mol2 -c bcc -at gaff2 -nc 0 -rn ETH -pf y > antechamber.log 2>&1
!parmchk2 -i ethanol.mol2 -f mol2 -o ethanol.frcmod -s gaff2
!cat ethanol.frcmod

tleap_in = """\
source leaprc.gaff2
loadamberparams ethanol.frcmod
ETH = loadmol2 ethanol.mol2
check ETH
saveamberparm ETH ethanol.prmtop ethanol.inpcrd
quit
"""
with open("tleap.in", "w") as f:
    f.write(tleap_in)
!tleap -f tleap.in > tleap.log 2>&1
!grep "Exiting LEaP" tleap.log

# ParmEd: prmtop → GROMACS; molfile выносит [ moleculetype ] и всё, что ниже, в ethanol.itp
eth = pmd.load_file("ethanol.prmtop", "ethanol.inpcrd")
q_rest = sum(a.charge for a in eth.atoms)         # antechamber округляет заряды до 1e-6, сумма 0.000001 e
eth.atoms[2].charge -= q_rest                     # остаток снимаем с O1: иначе на 200 молекулах набегает ~2e-4 e и grompp предупреждает о заряженной системе в PME
pmd.gromacs.GromacsTopologyFile.from_structure(eth).write("ethanol_gaff.top", molfile="ethanol.itp")
eth.save("ethanol.gro", overwrite=True)

# Из ethanol_gaff.top берём только блок [ atomtypes ]: [ defaults ] даст amber99sb-ildn.ff
top_text = open("ethanol_gaff.top").read()
i0 = top_text.index("[ atomtypes ]")
i1 = top_text.index("#include", i0)
with open("atomtypes_eth.itp", "w") as f:
    f.write(top_text[i0:i1].rstrip() + "\n")
!cat atomtypes_eth.itp

# Типы GAFF2 и заряды AM1-BCC
print(f"\n{'атом':5s} {'тип':4s} {'заряд, e':>9s}")
for a in eth.atoms:
    print(f"{a.name:5s} {a.type:4s} {a.charge:9.4f}")
print(f"Сумма зарядов: {sum(a.charge for a in eth.atoms):+.1e} e (поправка на O1: {-q_rest:+.0e} e)")

In [ ]:
# Боксы для пяти составов: ребро L = 2.8 нм, число молекул из плотности идеальной смеси, topol.top в каждом каталоге
X_LIST  = [0.0, 0.25, 0.5, 0.75, 1.0]  # мольная доля этанола
L       = 2.8                          # нм, ребро куба: L/2 = 1.4 нм > отсечки 1.0 нм при любом составе
M_ETH, M_WAT     = 46.069, 18.015      # г/моль
RHO_ETH, RHO_WAT = 785.1, 997.0        # кг/м³ при 298.15 K, чистые жидкости
N_A = 6.02214076e23

# Одна молекула воды (геометрия SPC/E из spc216.gro, O в начале координат); её размножит insert-molecules
water_gro = """\
SPC/E water
    3
    1SOL     OW    1   0.000   0.000   0.000
    1SOL    HW1    2  -0.093  -0.002   0.037
    1SOL    HW2    3   0.001  -0.039  -0.092
   1.00000   1.00000   1.00000
"""
with open("water.gro", "w") as f:
    f.write(water_gro)

box = f"{L:.3f}"                       # IPython не понимает {L:.3f} внутри !-команд, форматируем заранее
print(f"{'каталог':8s} {'x_EtOH':>7s} {'N_EtOH':>7s} {'N_H2O':>6s} {'атомов':>7s}")
for x in X_LIST:
    d = f"x{x:.2f}"
    os.makedirs(d, exist_ok=True)
    os.chdir(d)                        # команды ниже выполняются в каталоге состава

    # Мольный объём идеальной смеси (см³/моль) → полное число молекул в объёме L³
    v_id = x * M_ETH / RHO_ETH * 1e3 + (1 - x) * M_WAT / RHO_WAT * 1e3
    n_tot = round(L**3 * 1e-21 * N_A / v_id)
    n_eth = round(x * n_tot)
    n_wat = n_tot - n_eth

    # Сначала этанол в пустой бокс, затем вода в оставшееся место; обе компоненты распределены случайно
    if n_wat == 0:
        !{GMX} insert-molecules -box {box} {box} {box} -ci ../ethanol.gro -nmol {n_eth} -try 100 -o box.gro > insert_ethanol.log 2>&1
    elif n_eth == 0:
        !{GMX} insert-molecules -box {box} {box} {box} -ci ../water.gro -nmol {n_wat} -try 100 -o box.gro > insert_water.log 2>&1
    else:
        !{GMX} insert-molecules -box {box} {box} {box} -ci ../ethanol.gro -nmol {n_eth} -try 100 -o box_eth.gro > insert_ethanol.log 2>&1
        !{GMX} insert-molecules -f box_eth.gro -ci ../water.gro -nmol {n_wat} -try 100 -o box.gro > insert_water.log 2>&1
    n_atoms = int(open("box.gro").readlines()[1])
    assert n_atoms == 9 * n_eth + 3 * n_wat, f"{d}: вставлены не все молекулы"

    mols = (f"ETH   {n_eth}\n" if n_eth else "") + (f"SOL   {n_wat}\n" if n_wat else "")
    topol_top = f"""\
#include "amber99sb-ildn.ff/forcefield.itp"   ; [ defaults ] как в GAFF: comb-rule 2, fudgeLJ 0.5, fudgeQQ 0.8333
#include "../atomtypes_eth.itp"              ; типы GAFF2 этанола: c3, oh, hc, h1, ho
#include "../ethanol.itp"                    ; [ moleculetype ] ETH: заряды AM1-BCC, связи, углы, торсионы
#include "amber99sb-ildn.ff/spce.itp"        ; вода SPC/E, геометрию держит SETTLE

[ system ]
ethanol-water x = {x:.2f}

[ molecules ]
{mols}"""
    with open("topol.top", "w") as f:
        f.write(topol_top)

    print(f"{d:8s} {x:7.2f} {n_eth:7d} {n_wat:6d} {n_atoms:7d}")
    os.chdir("..")                     # обратно в каталог ethanol

### 3.2.2 Моделирование

Рассчитываем пять составов с мольной долей этанола $x$ = 0, 0.25, 0.5, 0.75, 1, каждый в своём подкаталоге

In [ ]:
# Параметры расчёта
em_mdp = """\
define        = -DFLEXIBLE   ; гибкая вода только на минимизации, как в 3.1
integrator    = steep        ; наискорейший спуск
emtol         = 100.0        ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep        = 0.01         ; начальный шаг, нм
nsteps        = 5000         ; предельное число шагов

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм; меньше L/2 = 1.4 нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям
"""

nvt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: связи с водородом этанола зафиксированы, вода жёсткая (SETTLE)
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = no           ; первый шаг динамики: стартовое состояние из em.gro
constraints   = h-bonds      ; длины связей C-H и O-H этанола фиксированы, без этого шаг 2 фс неустойчив
constraint-algorithm = lincs ; алгоритм LINCS для связей этанола; воду держит SETTLE из spce.itp

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; одна группа: этанол и вода вместе
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K
pcoupl        = no           ; объём фиксирован, ансамбль NVT

gen-vel       = yes          ; стартовые скорости из распределения Максвелла
gen-temp      = 298.15       ; при температуре 298.15 K
gen-seed      = -1           ; seed для случайных чисел
"""

npt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 10000        ; 10000 × 2 фс = 20 пс: баростат подстраивает объём под 1 бар

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии, объём и плотность в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = yes          ; продолжение NVT: скорости и состояние термостата из nvt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, стохастическое масштабирование ячейки; годится и для уравновешивания
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; 1/бар; у этанола ~1.1e-4, но величина влияет только на скорость подстройки объёма, не на среднее

gen-vel       = no           ; скорости не генерируем, они уже в nvt.cpt
"""

md_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 100000       ; 100000 × 2 фс = 200 пс, весь расчёт продуктивный

nstxout-compressed = 500     ; кадр в md.xtc каждую 1 пс
nstenergy     = 100          ; энергии, T, P, объём и плотность в .edr каждые 0.2 пс: 1000 точек для средних
nstlog        = 5000         ; строка в .log каждые 10 пс

continuation  = yes          ; продолжение NPT: скорости, термостат и баростат из npt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, тот же, что при уравновешивании
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; как при уравновешивании

gen-vel       = no           ; скорости не генерируем, они уже в npt.cpt
"""

for name, text in [("em.mdp", em_mdp), ("nvt.mdp", nvt_mdp), ("npt.mdp", npt_mdp), ("md.mdp", md_mdp)]:
    with open(name, "w") as f:
        f.write(text)

In [ ]:
# Протокол для всех составов: минимизация → NVT 20 пс → NPT 20 пс → NPT 200 пс
X_LIST = [0.0, 0.25, 0.5, 0.75, 1.0]

for x in X_LIST:
    d = f"x{x:.2f}"
    os.chdir(d)
    print(f"--- {d}")

    # Минимизация энергии
    !{GMX} grompp -f ../em.mdp -c box.gro -p topol.top -o em.tpr > grompp_em.out 2>&1
    !{GMX} mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1

    # NVT уравновешивание
    !{GMX} grompp -f ../nvt.mdp -c em.gro -p topol.top -o nvt.tpr > grompp_nvt.out 2>&1
    !{GMX} mdrun -deffnm nvt -ntomp 4 > mdrun_nvt.out 2>&1

    # NPT уравновешивание
    !{GMX} grompp -f ../npt.mdp -c nvt.gro -t nvt.cpt -p topol.top -o npt.tpr > grompp_npt.out 2>&1
    !{GMX} mdrun -deffnm npt -ntomp 4 > mdrun_npt.out 2>&1

    # NPT моделирование
    !{GMX} grompp -f ../md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr > grompp_md.out 2>&1
    !{GMX} mdrun -deffnm md -ntomp 4 > mdrun_md.out 2>&1

    os.chdir("..")

Объём бокса и потенциальная энергия зависят от числа молекул, которое различается для разных составов. Поэтому нормируем их на общее число молекул $N = N_{EtOH} + N_{H_2O}$, получая мольный объём $V_m = VN_A/N$ и потенциальную энергию на молекулу $U/N$.

Для идеальной смеси обе величины линейно зависят от мольной доли этанола:

$$Y_{ид}(x) = (1-x)Y_W + xY_{EtOH}$$

Отклонение от этой зависимости описывается уравнением Редлиха - Кистера:

$$Y(x) = (1-x)Y_W + xY_{EtOH} + x(1-x)[A_0 + A_1(2x-1)].$$

Отрицательный $A_0$ для $V_m$ указывает на уменьшение объёма при смешивании, а положительный $A_0$ для $U/N$ на энергетически менее выгодные межмолекулярные взаимодействия по сравнению с чистыми жидкостями.

Плотность рассчитываем из мольного объёма:

$$\rho(x) = \frac{M(x)}{V_m(x)}, \qquad M(x) = (1-x)M_{H_2O} + xM_{EtOH}$$

Давление баростат поддерживает на уровне 1 бар независимо от состава. Средние значения на графике совпадают с заданным давлением в пределах статистической погрешности.

In [ ]:
# Средние по продуктивному расчёту 200 пс по составам; T, P, ρ по времени, справа распределения
X_LIST = [0.0, 0.25, 0.5, 0.75, 1.0]
TERMS  = ["Potential", "Temperature", "Pressure", "Volume", "Density"]

avg, series = {}, {}                                 # avg[x][терм] = (среднее, Err.Est.); series[x] = столбцы md_energy.xvg
for x in X_LIST:
    d = f"x{x:.2f}"
    os.chdir(d)
    !printf "Potential\nTemperature\nPressure\nVolume\nDensity\n\n" | {GMX} energy -f md.edr -o md_energy.xvg > energy_md.out 2>&1
    avg[x] = {}
    for line in open("energy_md.out"):               # строки таблицы: терм, среднее, Err.Est., RMSD, дрейф
        parts = line.split()
        if parts and parts[0] in TERMS:
            avg[x][parts[0]] = (float(parts[1]), float(parts[2]))
    series[x] = np.loadtxt("md_energy.xvg", comments=("#", "@")).T   # t, U, T, P, V, ρ
    os.chdir("..")

fig, axes = plt.subplots(3, 2, figsize=(12.5, 6.5), constrained_layout=False,
                         gridspec_kw={"width_ratios": [6, 1], "wspace": 0})
(ax1, d1), (ax2, d2), (ax3, d3) = axes
for ax in (ax2, ax3):
    ax.sharex(ax1)
for ax, d in axes:
    d.sharey(ax)
    for a in (ax, d):
        a.set_prop_cycle(color=plt.cm.viridis(np.linspace(0, 0.9, len(X_LIST))))   # свой цвет на каждый состав

for x in X_LIST:
    t, U, T, P, V, rho = series[x]
    ax1.plot(t, T, lw=0.6, label=f"x = {x:.2f}")
    hill(d1, T)
    ax2.plot(t, P, lw=0.6)
    hill(d2, P)
    ax3.plot(t, rho, lw=0.6)
    hill(d3, rho)

ax1.axhline(298.15, color="0.3", lw=0.8, ls="--")   # целевая температура термостата
ax1.set_ylabel("T, K")
ax1.set_xlim(0, t[-1])
ax1.legend(ncol=len(X_LIST), frameon=False, loc="upper right")
ax2.axhline(1, color="0.3", lw=0.8, ls="--")        # целевое давление баростата
ax2.set_ylabel("P, бар")
ax3.set_ylabel("ρ, кг/м³")
ax3.set_xlabel("t, пс")

for ax in (ax1, ax2):
    ax.tick_params(labelbottom=False)
plt.show()

In [ ]:
# Средние по составам с ошибками Err.Est. из gmx energy; фит V_m и U/N по Редлиху-Кистеру, кривая ρ из фита V_m
N_A = 6.02214076e23
M_ETH, M_WAT = 46.069, 18.015                         # г/моль

n_mol = np.array([sum(int(n) for n in open(f"x{x:.2f}/topol.top").read().split("[ molecules ]")[1].split()[1::2])
                  for x in X_LIST])

x = np.array(X_LIST)
rho, rho_err = np.array([avg[x]["Density"] for x in X_LIST]).T
P, P_err     = np.array([avg[x]["Pressure"] for x in X_LIST]).T
V, V_err     = np.array([avg[x]["Volume"] for x in X_LIST]).T * 1e-21 * N_A / n_mol     # см³/моль
U, U_err     = np.array([avg[x]["Potential"] for x in X_LIST]).T / n_mol               # кДж/моль на молекулу

# Y(x) = (1-x)·Y_W + x·Y_EtOH + x(1-x)·[A0 + A1(2x-1)]: прямая между чистыми компонентами плюс избыточная часть
def redlich_kister(x, y_w, y_eth, a0, a1):
    return (1 - x) * y_w + x * y_eth + x * (1 - x) * (a0 + a1 * (2 * x - 1))

xx = np.linspace(0, 1, 200)
pV, covV = scipy.optimize.curve_fit(redlich_kister, x, V, sigma=V_err, absolute_sigma=True, p0=[V[0], V[-1], 0, 0])
pU, covU = scipy.optimize.curve_fit(redlich_kister, x, U, sigma=U_err, absolute_sigma=True, p0=[U[0], U[-1], 0, 0])
for name, p, cov, unit in [("V_m", pV, covV, "см³/моль"), ("U/N", pU, covU, "кДж/моль")]:
    e = np.sqrt(np.diag(cov))
    print(f"{name}: Y_W = {p[0]:.3f} ± {e[0]:.3f}, Y_EtOH = {p[1]:.3f} ± {e[1]:.3f}, "
          f"A0 = {p[2]:.2f} ± {e[2]:.2f}, A1 = {p[3]:.2f} ± {e[3]:.2f} {unit}")

fig, axes = plt.subplots(2, 2, figsize=(12.5, 6.5), sharex=True)
fits = [(axes[0, 0], rho, rho_err, "ρ, кг/м³",        ((1 - xx) * M_WAT + xx * M_ETH) / redlich_kister(xx, *pV) * 1e3),
        (axes[0, 1], V,   V_err,   "V$_m$, см³/моль", redlich_kister(xx, *pV)),
        (axes[1, 0], U,   U_err,   "U / N, кДж/моль", redlich_kister(xx, *pU)),
        (axes[1, 1], P,   P_err,   "P, бар",          None)]
for ax, y, err, label, fit in fits:
    ax.errorbar(X_LIST, y, yerr=err, fmt="o", color="black", ms=5, capsize=3, lw=1)
    if fit is not None:
        ax.plot(xx, fit, color="#D64933", lw=1)
    ax.set_ylabel(label)
axes[1, 1].axhline(1, color="#D64933", lw=0.8, ls="--")   # целевое давление баростата
for ax in axes[1]:
    ax.set_xlabel("x$_{EtOH}$")
    ax.set_xticks(X_LIST)
plt.show()

# 🧬 Глава 4. Моделирование олигопептидов

У дипептида аланина (ALA-ALA) есть две пептидая связь, и конформация остова задаётся двугранными углами: φ (C-N-Cα-C) и ψ (N-Cα-C-N). Структуру строим из последовательности программой ``tleap`` из пакета ``AmberTools``: команда `sequence` соединяет остатки из библиотеки силового поля в вытянутую цепь (φ = ψ = 180°). Топологию затем строит `pdb2gmx` в силовом поле AMBER99SB-ILDN с водой TIP3P, для которой это поле параметризовано.

## 4.1 Дипептид аланина 

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_8")
work_path = os.path.join(main_path, "ala2")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

### 4.1.1 Подготовка системы

In [ ]:
# Дипептид аланина из последовательности
SEQ = "NALA CALA"

tleap_in = f"""\
source leaprc.protein.ff14SB
pep = sequence {{ {SEQ} }}
savepdb pep ala2.pdb
quit
"""
with open("tleap.in", "w") as f:
    f.write(tleap_in)
!tleap -f tleap.in > tleap.log 2>&1
!grep "Exiting LEaP" tleap.log

In [ ]:
show("ala2.pdb")

In [ ]:
# Топология AMBER99SB-ILDN: pdb2gmx берёт параметры остатков из базы силового поля, водороды ставятся снова (-ignh)
!{GMX} pdb2gmx -f ala2.pdb -o ala2.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh > pdb2gmx.log 2>&1
!grep -E "^(WARNING|Error|Fatal)|Total charge|Now there are" -A 2 pdb2gmx.log

In [ ]:
# Кубический бокс: пептид не ближе 1.2 нм к граням
!{GMX} editconf -f ala2.gro -o box.gro -c -d 1.2 -bt cubic > editconf.log 2>&1
!{GMX} solvate -cp box.gro -cs spc216.gro -p topol.top -o solv.gro > solvate.log 2>&1
!grep "Number of solvent" solvate.log

In [ ]:
show("solv.gro")

### 4.1.2 Моделирование 

In [ ]:
# Минимизация энергии
em_mdp = """\
define        = -DFLEXIBLE   ; гибкая вода только на минимизации, как в 3.1
integrator    = steep        ; наискорейший спуск
emtol         = 100.0        ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep        = 0.01         ; начальный шаг, нм
nsteps        = 5000         ; предельное число шагов

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

!{GMX} grompp -f em.mdp -c solv.gro -p topol.top -o em.tpr > grompp_em.out 2>&1
!{GMX} mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1

In [ ]:
# Уравновешивание NVT
nvt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: связи с водородом зафиксированы
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = no           ; первый шаг динамики: стартовое состояние из em.gro
constraints   = h-bonds      ; длины связей X-H пептида фиксированы; воду держит SETTLE из tip3p.itp
constraint-algorithm = lincs ; алгоритм LINCS для связей пептида

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; одна группа: в пептиде 23 атома, отдельный термостат для него слишком шумный
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K
pcoupl        = no           ; объём фиксирован, ансамбль NVT

gen-vel       = yes          ; стартовые скорости из распределения Максвелла
gen-temp      = 298.15       ; при температуре 298.15 K
gen-seed      = -1           ; seed для случайных чисел
"""
with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp)

!{GMX} grompp -f nvt.mdp -c em.gro -p topol.top -o nvt.tpr > grompp_nvt.out 2>&1
!{GMX} mdrun -deffnm nvt -ntomp 4 > mdrun_nvt.out 2>&1

In [ ]:
# Уравновешивание NPT
npt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии, объём и плотность в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = yes          ; продолжение NVT: скорости и состояние термостата из nvt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, стохастическое масштабирование ячейки; годится и для уравновешивания
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости не генерируем, они уже в nvt.cpt
"""
with open("npt.mdp", "w") as f:
    f.write(npt_mdp)

!{GMX} grompp -f npt.mdp -c nvt.gro -t nvt.cpt -p topol.top -o npt.tpr > grompp_npt.out 2>&1
!{GMX} mdrun -deffnm npt -ntomp 4 > mdrun_npt.out 2>&1

In [ ]:
# Моделирование NPT
md_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 500000       ; 500000 × 2 фс = 1 нс

nstxout-compressed = 500     ; кадр в md.xtc каждую 1 пс: 1000 кадров для карты Рамачандрана
nstenergy     = 500          ; энергии, T, P, объём и плотность в .edr каждую 1 пс
nstlog        = 5000         ; строка в .log каждые 10 пс

continuation  = yes          ; продолжение NPT: скорости, термостат и баростат из npt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, тот же, что при уравновешивании
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости не генерируем, они уже в npt.cpt
"""
with open("md.mdp", "w") as f:
    f.write(md_mdp)

!{GMX} grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr > grompp_md.out 2>&1
!{GMX} mdrun -deffnm md -ntomp 4 > mdrun_md.out 2>&1

In [ ]:
# Траектория пептида
show("solv.gro", "md.xtc", stride=50, cmd="show sticks")

In [ ]:
# Траектория одного пептида из траектории с водой
!printf "Protein\nProtein\n" | {GMX} trjconv -s md.tpr -f md.xtc -pbc mol -center -o md_mol.xtc > trjconv.log 2>&1

In [ ]:
show("ala2.gro", "md_mol.xtc", stride=10, cmd="show sticks; orient")

### 4.1.3 Структура и динамика пептида 

In [ ]:
# RMSD остова от стартовой структуры; углы ψ₁ (N₁-Cα₁-C₁-N₂) и φ₂ (C₁-N₂-Cα₂-C₂) вокруг пептидной связи Ala1-Ala2
!printf "Backbone\nBackbone\n" | {GMX} rms -s ala2.gro -f md_mol.xtc -o rmsd.xvg -tu ns > rms.log 2>&1

# Номера атомов по именам из ala2.gro; в .ndx одна группа из двух четвёрок атомов - два двугранных угла
gro = open("ala2.gro").read().splitlines()
n = int(gro[1])
idx = {(int(l[0:5]), l[10:15].strip()): i + 1 for i, l in enumerate(gro[2:2 + n])}   # (остаток, атом) → номер
psi1 = [idx[1, "N"], idx[1, "CA"], idx[1, "C"], idx[2, "N"]]
phi2 = [idx[1, "C"], idx[2, "N"], idx[2, "CA"], idx[2, "C"]]
with open("dihedrals.ndx", "w") as f:
    f.write("[ psi1_phi2 ]\n" + " ".join(map(str, psi1 + phi2)) + "\n")

!echo 0 | {GMX} angle -f md_mol.xtc -n dihedrals.ndx -type dihedral -all -ov dihedrals.xvg > angle.log 2>&1

In [ ]:
t, rmsd = np.loadtxt("rmsd.xvg", comments=("#", "@")).T                     # t в нс, RMSD в нм
_, _, psi, phi = np.loadtxt("dihedrals.xvg", comments=("#", "@")).T        # столбцы: время в пс, среднее, ψ₁, φ₂

fig = plt.figure(figsize=(12.5, 2.86))
outer = fig.add_gridspec(1, 2, width_ratios=[3, 1], wspace=0.2)
left = outer[0].subgridspec(2, 1, height_ratios=[0.25, 1], hspace=0)      # та же разметка по высоте, что справа
inner = outer[1].subgridspec(2, 2, width_ratios=[1, 0.25], height_ratios=[0.25, 1], wspace=0, hspace=0)
ax_r = fig.add_subplot(left[1])                      # RMSD - на высоту 2D-распределения
ax_j = fig.add_subplot(inner[1, 0])                  # (φ₂, ψ₁)
ax_x = fig.add_subplot(inner[0, 0], sharex=ax_j)     # распределение φ₂
ax_y = fig.add_subplot(inner[1, 1], sharey=ax_j)     # распределение ψ₁

ax_r.plot(t, rmsd * 10, color="black", lw=0.5)
ax_r.set_xlabel("t, нс")
ax_r.set_ylabel("RMSD, Å")
ax_r.set_xlim(0, t[-1])
ax_r.set_ylim(0, 1)


bins = np.linspace(-180, 180, 73)                    # ячейки по 5°
sns.kdeplot(x=phi, y=psi, ax=ax_j, fill=True, cmap=sns.light_palette("#DC602E", as_cmap=True),
            levels=12, thresh=0.02, bw_adjust=0.7)
sns.histplot(x=phi, ax=ax_x, bins=bins, kde=True, color="#DC602E", edgecolor=None)
sns.histplot(y=psi, ax=ax_y, bins=bins, kde=True, color="#DC602E", edgecolor=None)

ax_j.set_xlim(-180, 180)
ax_j.set_ylim(-180, 180)
ax_j.set_xticks([-180, -90, 0, 90, 180])
ax_j.set_yticks([-180, -90, 0, 90, 180])
ax_j.set_xlabel("φ$_2$, °")
ax_j.set_ylabel("ψ$_1$, °")
for ax in (ax_x, ax_y):
    ax.set_axis_off()
plt.show()

## 4.2 Циклический олигопептид 

Гормон окситоцин состоит из девяти остатков: Cys-Tyr-Ile-Gln-Asn-Cys-Pro-Leu-Gly-NH₂. Дисульфидная связь Cys1–Cys6 замыкает шесть остатков в кольцо из 20 атомов, а «хвост» Pro7-Leu8-Gly9 остаётся снаружи. C-конец амидирован. Стартовую структуру берём из PDB 2MGO: это ансамбль из 20 конформеров, полученный по данным ЯМР в воде (90 % H₂O, pH 3.5, 298 K [Koehbach et al., PNAS 2013](https://doi.org/10.1073/pnas.1311183110)
). Используем первую, наиболее представленную в ансамбле модель. Вместе со структурой скачиваем файл ограничений `2mgo.mr`: 49 межпротонных расстояний.

При pH 3.5 N-концевая аминогруппа протонирована, а кислых групп в окситоцине нет, поэтому заряд пептида +1. Для нейтральности бокса добавляем на один ион Cl⁻ больше.

### 4.2.1 Подготовка системы

In [ ]:
main_path = os.path.join(REPO, "Занятия", "Занятие_8")
work_path = os.path.join(main_path, "oxytocin")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

In [ ]:
# Окситоцин из PDB 2MGO: первая модель ЯМР-ансамбля без водородов; C-концевой амид - остаток NHE вместо атома OXT
import urllib.request

NAME = "2MGO"

urllib.request.urlretrieve("https://files.rcsb.org/download/2MGO.pdb", f"{NAME}.pdb")

# В 2MGO C-конец записан как свободная COOH, у природного окситоцина он амидирован:
# атом OXT становится азотом амида NH2 (остаток NHE силового поля Amber), водороды достроит pdb2gmx
lines = []
for line in open(f"{NAME}.pdb"):
    if line.startswith("ENDMDL"):                    # конец первой модели
        break
    if not line.startswith("ATOM") or line[76:78].strip() == "H":
        continue
    if line[12:16].strip() == "OXT":
        line = line[:12] + " N   NHE A  10" + line[26:76] + " N" + line[78:]
    lines.append(line)
with open("oxytocin.pdb", "w") as f:
    f.write("".join(lines) + "END\n")

In [ ]:
show("oxytocin.pdb")

In [ ]:
# Топология: pdb2gmx сам находит пару атомов SG на расстоянии связи и образует дисульфид Cys1-Cys6
!{GMX} pdb2gmx -f oxytocin.pdb -o oxytocin.gro -p topol.top -ff amber99sb-ildn -water tip3p -ignh > pdb2gmx.log 2>&1

# Кубический бокс: пептид не ближе 1.2 нм к граням
!{GMX} editconf -f oxytocin.gro -o box.gro -c -d 1.2 -bt cubic > editconf.log 2>&1
!{GMX} solvate -cp box.gro -cs spc216.gro -p topol.top -o solv.gro > solvate.log 2>&1
!grep "Number of solvent" solvate.log

ions_mdp = """\
integrator    = steep        ; ions.tpr нужен только genion, mdrun по нему не запускаем
nsteps        = 0
cutoff-scheme = Verlet       ; список соседей Верле с буфером
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("ions.mdp", "w") as f:
    f.write(ions_mdp)

!{GMX} grompp -f ions.mdp -c solv.gro -p topol.top -o ions.tpr -maxwarn 1 > grompp_ions.out 2>&1
!echo SOL | {GMX} genion -s ions.tpr -o ions.gro -p topol.top -pname NA -nname CL -conc 0.15 -neutral > genion.log 2>&1
!grep -E "^Will try" genion.log

In [ ]:
show("ions.gro")

### 4.2.2 Моделирование

In [ ]:
# Минимизация энергии
em_mdp = """\
define        = -DFLEXIBLE   ; гибкая вода только на минимизации
integrator    = steep        ; наискорейший спуск
emtol         = 100.0        ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep        = 0.01         ; начальный шаг, нм
nsteps        = 5000         ; предельное число шагов

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

!{GMX} grompp -f em.mdp -c ions.gro -p topol.top -o em.tpr > grompp_em.out 2>&1
!{GMX} mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1

In [ ]:
# Уравновешивание NVT
nvt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс: связи с водородом зафиксированы
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = no           ; первый шаг динамики: стартовое состояние из em.gro
constraints   = h-bonds      ; длины связей X-H пептида фиксированы; воду держит SETTLE из tip3p.itp
constraint-algorithm = lincs ; алгоритм LINCS для связей пептида

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси
tc-grps       = System       ; одна группа: пептид, вода и ионы вместе
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K - как в ЯМР-эксперименте
pcoupl        = no           ; объём фиксирован, ансамбль NVT

gen-vel       = yes          ; стартовые скорости из распределения Максвелла
gen-temp      = 298.15       ; при температуре 298.15 K
gen-seed      = -1           ; seed для случайных чисел
"""
with open("nvt.mdp", "w") as f:
    f.write(nvt_mdp)

!{GMX} grompp -f nvt.mdp -c em.gro -p topol.top -o nvt.tpr > grompp_nvt.out 2>&1
!{GMX} mdrun -deffnm nvt -ntomp 4 > mdrun_nvt.out 2>&1

In [ ]:
# Уравновешивание NPT: 20 пс при 298.15 K и 1 бар, баростат подстраивает объём бокса
npt_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 10000        ; 10000 × 2 фс = 20 пс

nstxout-compressed = 0       ; траекторию не сохраняем
nstenergy     = 100          ; энергии, объём и плотность в .edr каждые 0.2 пс
nstlog        = 1000         ; строка в .log каждые 2 пс

continuation  = yes          ; продолжение NVT: скорости и состояние термостата из nvt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; одна группа: пептид, вода и ионы вместе
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, стохастическое масштабирование ячейки; годится и для уравновешивания
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости не генерируем, они уже в nvt.cpt
"""
with open("npt.mdp", "w") as f:
    f.write(npt_mdp)

!{GMX} grompp -f npt.mdp -c nvt.gro -t nvt.cpt -p topol.top -o npt.tpr > grompp_npt.out 2>&1
!{GMX} mdrun -deffnm npt -ntomp 4 > mdrun_npt.out 2>&1

In [ ]:
# Продуктивный расчёт NPT: 1 нс при 298.15 K и 1 бар; в md.xtc вся система
md_mdp = """\
integrator    = md           ; leap-frog
dt            = 0.002        ; 2 фс
nsteps        = 500000       ; 500000 × 2 фс = 1 нс

nstxout-compressed = 500     ; кадр в md.xtc каждую 1 пс: 1000 кадров для сравнения с ЯМР
nstenergy     = 500          ; энергии, T, P, объём и плотность в .edr каждую 1 пс
nstlog        = 5000         ; строка в .log каждые 10 пс

continuation  = yes          ; продолжение NPT: скорости, термостат и баростат из npt.cpt
constraints   = h-bonds      ; как в NVT
constraint-algorithm = lincs ; как в NVT

cutoff-scheme = Verlet       ; список соседей Верле с буфером
nstlist       = 10           ; список соседей обновляется каждые 10 шагов
coulombtype   = PME          ; электростатика по Эвальду
rcoulomb      = 1.0          ; радиус отсечки прямой части PME, нм
fourierspacing = 0.12        ; шаг сетки PME, нм
pme-order     = 4            ; кубическая интерполяция зарядов на сетку
vdwtype       = Cut-off      ; ЛД отсекается на rvdw
rvdw          = 1.0          ; радиус отсечки ЛД, нм
DispCorr      = EnerPres     ; хвостовая поправка к энергии и давлению
pbc           = xyz          ; периодические граничные условия по трём осям

tcoupl        = V-rescale    ; термостат Бусси, тот же, что в NVT
tc-grps       = System       ; термостатируемая группа - все атомы
tau-t         = 0.1          ; время связи с термостатом, пс
ref-t         = 298.15       ; целевая температура, K

pcoupl          = C-rescale  ; баростат, тот же, что при уравновешивании
pcoupltype      = isotropic  ; бокс масштабируется одинаково по трём осям
tau-p           = 2.0        ; время связи с баростатом, пс
ref-p           = 1.0        ; целевое давление, бар
compressibility = 4.5e-5     ; изотермическая сжимаемость воды при 298 K, 1/бар

gen-vel       = no           ; скорости не генерируем, они уже в npt.cpt
"""
with open("md.mdp", "w") as f:
    f.write(md_mdp)

!{GMX} grompp -f md.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr > grompp_md.out 2>&1
!{GMX} mdrun -deffnm md -ntomp 4 > mdrun_md.out 2>&1

In [ ]:
# Траектория окситоцина в растворе: структура и траектория всей системы, 50 кадров
show("md.gro", "md.xtc", stride=20, cmd="show sticks; orient")

In [ ]:
# Траектория молекулы окситоцина (без воды и ионов)
!echo Protein | {GMX} convert-tpr -s md.tpr -o md_prot.tpr > convert.log 2>&1
!printf "Protein\nProtein\n" | {GMX} trjconv -s md.tpr -f md.xtc -pbc mol -center -o md_mol.xtc > trjconv.log 2>&1

In [ ]:
show("oxytocin.gro", "md_mol.xtc", stride=10, cmd="show sticks; orient")

RMSD (§ 1.6.2 занятие 7) характеризует отклонение структуры в момент $t$ от отсчётной, усреднённое по атомам, в то время как **RMSF** характеризует амплитуду флуктуаций атома $i$ относительно его среднего положения, усреднённую по времени:

$$\text{RMSF}_i = \sqrt{\big\langle\, |\mathbf{r}_i(t) - \langle \mathbf{r}_i \rangle|^2 \big\rangle_t}$$

Перед расчётом каждый кадр совмещается с отсчётной структурой для исключения вращения и трансляции молекулы как целого. Значения RMSF по остаткам получают усреднением RMSF значений для атомов.

Профиль RMSF вдоль цепи используется для выявления конформационно подвижных участков белка. Элементы вторичной структуры и фрагменты, стабилизированные водородными или дисульфидными связями, характеризуются RMSF от 0.5 до 1 Å, а петли, концевые участки и экспонированные боковые цепи от 2 до 5 Å.

In [ ]:
# RMSD остова по времени и RMSF остова по остаткам
!printf "Backbone\nBackbone\n" | {GMX} rms -s md_prot.tpr -f md_mol.xtc -o rmsd.xvg -tu ns > rms.log 2>&1
!echo Backbone | {GMX} rmsf -s md_prot.tpr -f md_mol.xtc -res -o rmsf.xvg > rmsf.log 2>&1
!grep -E "^(Error|Fatal)" -A 3 rms.log rmsf.log

t, rmsd = np.loadtxt("rmsd.xvg", comments=("#", "@")).T     # t в нс, RMSD в нм
res, rmsf = np.loadtxt("rmsf.xvg", comments=("#", "@")).T   # номер остатка, RMSF в нм
RES = {1: "Cys", 2: "Tyr", 3: "Ile", 4: "Gln", 5: "Asn", 6: "Cys", 7: "Pro", 8: "Leu", 9: "Gly", 10: "NH₂"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 3.5), gridspec_kw={"width_ratios": [2, 1]})
ax1.plot(t, rmsd * 10, color="black", lw=0.6)
ax1.set_xlabel("t, нс")
ax1.set_ylabel("RMSD остова, Å")
ax1.set_xlim(0, t[-1])
ax2.bar(res, rmsf * 10, color=["#00BFFF" if r <= 6 else "#D64933" for r in res])
ax2.set_xticks(res)
ax2.set_xticklabels([f"{RES[int(r)]}{int(r)}" for r in res], rotation=90)
ax2.set_ylabel("RMSF остова, Å")
plt.tight_layout()
plt.show()

### 4.2.3 Сравнение с экспериментом

Ядерный эффект Оверхаузера (NOE, Nuclear Overhauser Effect) возникает вследствие взаимодействия между ядерными спинами через пространство (диполь-дипольное взаимодействие). В ЯМР-спектре он проявляется в виде кросс-пиков между сигналами ядер, пространственно близких друг к другу, даже если они не связаны непосредственно химическими связями.

Интенсивность сигнала NOE зависит от расстояния между ядрами приблизительно как $r^{-6}$. Эта зависимость имеет важное следствие: близкие межъядерные расстояния вносят существенно больший вклад в NOE, чем удалённые. Например, взаимодействие на расстоянии 3 Å примерно в 64 раза интенсивнее, чем на расстоянии 6 Å.

В случае гибких молекул или ансамбля конформаций сигнал NOE усредняется по времени и по всем молекулам образца. Поэтому экспериментальное расстояние не соответствует обычному среднему расстоянию между атомами, а определяется через усреднение величины $r^{-6}$:
$$
r_{\mathrm{eff}} =
\left\langle r^{-6}\right\rangle^{-1/6}
=
\left(\sum_i p_i r_i^{-6}\right)^{-1/6},
$$

где $r_i$ — расстояние между ядрами в конформации $i$, а $p_i$ — её относительная заселённость.

**Сопоставление с молекулярной динамикой**

При анализе траекторий молекулярной динамики NOE-расстояния можно рассчитывать непосредственно из ансамбля полученных структур, используя усреднение $r^{-6}$. Если рассчитанные эффективные расстояния согласуются с экспериментальными ограничениями, это подтверждает, что используемое силовое поле и протокол молекулярной динамики могут воспроизводить конформационное поведение биополимера.

In [ ]:
# Ограничения ЯМР из 2MGO.mr: NOE-расстояния
RES = {1: "Cys", 2: "Tyr", 3: "Ile", 4: "Gln", 5: "Asn", 6: "Cys", 7: "Pro", 8: "Leu", 9: "Gly", 10: "NH₂"}
noe_text = open(os.path.join(main_path, f"{NAME}.mr")).read().split("!dihedral restraints")[0]

noe = []
for line in noe_text.splitlines():
    if line.startswith("assign"):
        w = line.replace("(", " ").replace(")", " ").split() 
        noe.append((int(w[2]), w[5], int(w[7]), w[10], float(w[11]), float(w[11]) + float(w[13])))

In [ ]:
# NOE: эффективное расстояние ⟨Σr⁻⁶⟩⁻¹ᐟ⁶ по траектории ± стандартное отклонение против верхней границы ЯМР
!echo Protein | {GMX} trjconv -s md_prot.tpr -f md_mol.xtc -o md_mol.gro > trjconv_gro.log 2>&1
gro = open("md_mol.gro").read().splitlines()
n = int(gro[1])                                      # атомов в кадре; кадр = заголовок + число атомов + n строк + бокс
names = [(int(l[0:5]), l[10:15].strip()) for l in gro[2:2 + n]]
xyz = np.array([[[float(l[20:28]), float(l[28:36]), float(l[36:44])] for l in gro[f + 2:f + 2 + n]]
                for f in range(0, len(gro), n + 3)]) * 10

def atoms(resid, name):
    # Имена XPLOR → GROMACS: HN → H; «#» - все протоны группы; у Ile метил δ в GROMACS это CD с HD1-HD3
    if name == "HN":
        name = "H"
    if name == "HD1#" and RES[resid] == "Ile":
        name = "HD#"
    if name.endswith("#"):
        return [i for i, (r, a) in enumerate(names) if r == resid and a.startswith(name[:-1])]
    return [i for i, (r, a) in enumerate(names) if r == resid and a == name]

r_md, r_std = [], []
for r1, a1, r2, a2, d, up in noe:
    dist = np.linalg.norm(xyz[:, atoms(r1, a1), None] - xyz[:, None, atoms(r2, a2)], axis=-1)   # кадры × протоны 1 × протоны 2
    inv6 = np.sum(dist ** -6, axis=(1, 2))           # по кадрам; для групп протонов - сумма по парам
    r_md.append(np.mean(inv6) ** (-1 / 6))
    r_std.append(np.std(inv6 ** (-1 / 6)))           # разброс мгновенного расстояния по траектории
r_md, r_std = np.array(r_md), np.array(r_std)
upper = np.array([up for *_, up in noe])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5), gridspec_kw={"width_ratios": [2.3, 1]})
x = np.arange(len(noe))
ax1.bar(x - 0.2, upper, width=0.4, color="0.75")
ax1.bar(x + 0.2, r_md, width=0.4, color="#D64933", yerr=r_std, error_kw={"elinewidth": 0.8, "capsize": 1.5})
ax1.set_xticks(x)
ax1.set_xticklabels([f"{RES[r1]}{r1} {a1} – {RES[r2]}{r2} {a2}" for r1, a1, r2, a2, d, up in noe], rotation=90, fontsize=6)
ax1.set_xlim(-1, len(noe))
ax1.set_ylabel("r, Å")
ax1.legend(handles=[plt.Rectangle((0, 0), 1, 1, color="0.75", label="верхняя граница ЯМР"),
                    plt.Rectangle((0, 0), 1, 1, color="#D64933", label="МД, ⟨r⁻⁶⟩⁻¹ᐟ⁶ ± σ")],
           frameon=False, loc="upper left")

ax2.errorbar(upper, r_md, yerr=r_std, fmt="o", ms=5, color="#D64933", elinewidth=0.8, capsize=2)
ax2.plot([1.5, 9], [1.5, 9], color="0.4", lw=1, ls="--")   # r_МД = граница: выше линии - нарушение
ax2.set_xlim(1.5, 9)
ax2.set_ylim(1.5, 9)
ax2.set_aspect("equal")
ax2.set_xlabel("верхняя граница ЯМР, Å")
ax2.set_ylabel("МД, ⟨r⁻⁶⟩⁻¹ᐟ⁶, Å")
plt.tight_layout()
plt.show()

# 📌 Дополнительные материалы

**Визуализация структур и траекторий**:
* PyMOL
* Chimera
* ChimeraX

**Подготовка систем и топологий**:
* AmberTools
* ACPYPE 
* ParmEd

**Анализ траекторий**:
* gmx energy, rms, rmsf, gyrate, rdf, msd, hbond, rama, cluster* - встроенные инструменты GROMACS.
* MDAnalysis 
* MDTraj

**Экспериментальные данные для сравнения**:
* PDB (rcsb.org) - структуры биополимеров
* BMRB (bmrb.io) - данные ЯМР

**Документация и обучающие материалы**:
* GROMACS Manual
* GROMACS Tutorials